# T0.3 · A0–B2 SLAP2 ↔ HARP ↔ BonVision synchronization

**Scientific objective:** Demonstrate that SLAP2 acquisition-clock cadence is represented by HARP DI3, establish which acquisition epochs are represented by HARP, and show BonVision stimulus timing relative to the HARP photodiode. Compare both BV-encoded photodiode (piecewise) and early frame-modulo/affine sessions.

**Important identifiability limit:** DI3 is the **only** HARP-timestamped SLAP2 signal. Consequently, the green SLAP2 clock below is **reconstructed from `linesPerCycle / lineRateHz`, with its origin set equal to the first observed DI3 edge**. Green/pink coincidence at the anchor is imposed, not an independently measured hardware delay. Quantities labeled *anchored discrepancy*, *cadence difference*, or *drift* are testable; absolute SLAP2-to-DI3 edge lag is **not** separately measurable from these files.

**Epoch window:** HARP DI3 can cover a shorter interval than source acquisition. Pair accepted source and behavior epochs chronologically; do not interpret unequal whole-file pulse/cycle counts as missing cycles. Without a separate absolute SLAP2 start signal, the allocation of unmatched source duration to the beginning versus end is not independently known. Source/behavior processing's tail-clipping policy can nevertheless be documented and illustrated explicitly.

No physiological traces are loaded or filtered. All exported figures have data-source/method/interpretation/limitation notes. **A one-display-frame pass requires separately validated end-to-end timing**, not just cadence matching or low in-sample photodiode residuals.

In [ ]:
%matplotlib notebook

In [ ]:
from pathlib import Path
import sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

for parent in (Path.cwd(), *Path.cwd().parents):
    if (parent/'src'/'vip_slap2_analysis'/'metadata'/'t03_cycle_clock.py').is_file():
        sys.path.insert(0, str(parent/'src'))
        break
        
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.metadata.t03_cycle_clock import (
    collect_sessions, session_record, get_di3_edges, split_di3_epochs,
    check_epochs, fnum, read_json, SESSION_TYPES, harp_pd_di3_phase)

from vip_slap2_analysis.metadata.t03_event_metrics import event_pd_diagnostics
from vip_slap2_analysis.metadata.t03_session_sync import _load_pd_local, _pd_edges_local

from vip_slap2_analysis.utils.utils import save_figure

plt.rcParams.update({'font.size':10,'axes.spines.right':False,'axes.spines.top':False,
                      'figure.figsize':(10,4.5),'savefig.dpi':300})
print('Backend:',plt.get_backend())
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

## 0 · Configuration

The full dataset scan is registry-driven. By default, select A0–B2 sessions; use `SESSION_IDS` to narrow the first run. The three representative cases are **863774 voltage**, **834788 glutamate + Ca²⁺**, and **803496 early glutamate**. Set `FORCE_METADATA_REFRESH=True` if a source file or saved QC metadata changed. DI3 edge arrays are separately cached with source file size and modification-time invalidation.

Set `ANALYZE_ALL_PHOTODIODE=True` only after the fast clock audit; loading every analog photodiode pickle is the expensive part. `%matplotlib notebook` is supported in the classic notebook frontend; in JupyterLab use `%matplotlib widget` if your installation requires it.

In [ ]:
SUMMARY_XLSX = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\VIP_SD_summary.xlsx")
SESSION_IDS = None                       # None = all registry A0–B2 sessions
SESSION_TYPES_TO_INCLUDE = ['A0','A1','A2','B0','B1','B2']
REPRESENTATIVES = [
    '863774_2026-08-04_08-12-47',  # voltage
    '834788_2026-02-13_15-22-44',  # glutamate+Ca2+
    '803496_2025-07-31_09-43-28',  # early glutamate / legacy behavior
]
FORCE_METADATA_REFRESH = False
ANALYZE_ALL_PHOTODIODE = False           # slow; focus sessions always attempted
DISPLAY_HZ_REFERENCE = 60.0             # reference ONLY; use verified period for acceptance
OUTPUT = Path.cwd()/'T03_cycle_clock_outputs'
CACHE = OUTPUT/'cache'
for folder in (OUTPUT, CACHE, OUTPUT/'figures'):folder.mkdir(parents=True,exist_ok=True)
assert SUMMARY_XLSX.is_file(), f'Registry workbook not found: {SUMMARY_XLSX}'
print('Source:', SUMMARY_XLSX)
print('60-Hz reference frame:', 1000/DISPLAY_HZ_REFERENCE, 'ms')

## 1 · Fast registry inventory and evidence table

Uses the registry's session list but does **not** call `resolve_assets()` repeatedly (it recursively searches the full directory multiple times). Instead, scans each session directory once for canonical input files, loads lightweight metadata via `VoltageSummary` / `GlutamateSummary`, and caches DI3 edges. The source `.mat` must expose `linesPerCycle`, `lineRateHz`, and preferably per-epoch `nCycles`; older sessions with missing metadata are recorded as unmeasurable, not silently estimated.

In [ ]:
registry = VIPSessionRegistry.from_excel(SUMMARY_XLSX)
registry_rows = registry.sessions_df.copy()
selected_rows = registry_rows[registry_rows['session_type'].astype(str).str.upper().isin(SESSION_TYPES_TO_INCLUDE)]
if SESSION_IDS is not None:
    selected_rows=selected_rows[selected_rows['session_id'].astype(str).isin(SESSION_IDS)]
    if not len(selected_rows): raise ValueError('No requested sessions in A0–B2 registry')
print('Selected:',len(selected_rows),'sessions')
# Refresh forces reading source/QC metadata, not modifications of any session data.
df,_=collect_sessions(SUMMARY_XLSX,CACHE,session_ids=SESSION_IDS,
                      session_types=set(SESSION_TYPES_TO_INCLUDE),force=FORCE_METADATA_REFRESH)
df.to_csv(OUTPUT/'t03_clock_session_evidence.csv',index=False)
cols=['session_id','session_type','alignment_pathway','clock_relationship',
      'reference_dmd','lineRateHz','linesPerCycle','di3_period_ms','source_period_ms',
      'period_error_ppm','n_di3_edges','di3_n_epochs','di3_n_irregular_intervals',
      'harp_di3_duration_s','source_duration_s','duration_overhang_s','source_read_status']
display(df[[c for c in cols if c in df]].head(25))

In [ ]:
CAPTIONS=[]
FRAME_MS=1000/DISPLAY_HZ_REFERENCE

def missing(ax,label):
    ax.text(.5,.5,label,transform=ax.transAxes,ha='center',va='center',color='.45')
    ax.set_xticks([]);ax.set_yticks([])

def publish(fig,key,title,source,method,meaning,limit):
    fig.suptitle(title,fontsize=13,y=.995)
    fig.tight_layout(rect=(0,.055,1,.95))
    fig.text(.03,.015,'SOURCE: '+source[:190],fontsize=7,color='.35')
    fig.savefig(OUTPUT/'figures'/f'{key}.png',bbox_inches='tight',dpi=300,facecolor='white')
    fig.savefig(OUTPUT/'figures'/f'{key}.svg',bbox_inches='tight',facecolor='white')
    desc=f'SOURCE: {source}\nMETHOD: {method}\nINTERPRETATION: {meaning}\nLIMITATION: {limit}\n'
    (OUTPUT/'figures'/f'{key}.txt').write_text(desc)
    CAPTIONS.append(dict(figure=key,title=title,source=source,method=method,
                         interpretation=meaning,limitation=limit))
    plt.show()

def finite(d,col):return pd.to_numeric(d.get(col,pd.Series(np.nan,index=d.index)),errors='coerce')

def session_paths(sid):
    r=df[df.session_id.astype(str)==str(sid)]
    if r.empty:return None
    return r.iloc[0]

def focus(sid):
    r=selected_rows[selected_rows.session_id.astype(str)==str(sid)]
    if r.empty: return None,None
    return session_record(r.iloc[0],CACHE)

def edges_to_steps(edges,t0,t1,dt=.001):
    # plot pulse ticks as vector line segments, not an invented analog waveform
    e=np.asarray(edges,float)
    e=e[(e>=t0)&(e<=t1)]
    return e

## 2 · Does HARP DI3 encode the SLAP2 acquisition cycle?

We compare each session's observed DI3 period to the period implied by the SLAP2 source acquisition metadata: `T_cycle = linesPerCycle / lineRateHz`. If DI3 follows a different *integer-line cadence* (seen in some historical recordings), the relationship is classified separately; we do not force such sessions into the DMD-cycle category.

The following distributions include only sessions for which both clocks' period metadata are available.

In [ ]:
fig,axs=plt.subplots(1,3,figsize=(12,4))
class_counts=df['clock_relationship'].fillna('unavailable').value_counts()
axs[0].bar(class_counts.index.astype(str),class_counts.values,color='#507fa3')
axs[0].tick_params(axis='x',rotation=35);axs[0].set_ylabel('Sessions')
for ax,col,xlab in [(axs[1],'period_error_ppm','HARP DI3 vs nominal cycle (ppm)'),
                    (axs[2],'cadence_minus_cycle_lines','Effective lines/pulse − linesPerCycle')]:
    v=finite(df,col).dropna()
    if len(v):ax.hist(v,bins=min(25,max(5,int(np.sqrt(len(v))*2))),color='#507fa3',edgecolor='white');ax.axvline(0,c='black',ls='--',lw=1)
    else:missing(ax,'No comparable source metadata')
    ax.set_xlabel(xlab);ax.set_ylabel('Sessions')
publish(fig,'01_cycle_clock_evidence','SLAP2-to-HARP cycle-clock correspondence',
        'Session registry; SLAP2 SummaryLoCo/dendriticVoltageSummary acquisition metadata; HARP_df DI3',
        'Period from long contiguous DI3 epochs compared to linesPerCycle/lineRateHz; integer-line cadence classified separately.',
        'Distribution near zero supports that DI3 measures the expected SLAP2 acquisition cadence.',
        'A cadence match does not independently measure electrical latency or prove an absolute source start offset.')

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(10.7,4.4))
for typ,grp in df.groupby('session_type'):
    v=finite(grp,'period_error_ppm').dropna()
    if len(v):axs[0].scatter([typ]*len(v),v,s=16,alpha=.7,label=typ)
axs[0].axhline(0,color='black',ls='--',lw=1);axs[0].set_ylabel('Clock period error (ppm)')
for typ,grp in df.groupby('session_type'):
    v=finite(grp,'di3_interval_p95_jitter_us').dropna()
    if len(v):axs[1].scatter([typ]*len(v),v,s=16,alpha=.7)
axs[1].set_ylabel('DI3 p95 absolute interval jitter (µs)')
publish(fig,'02_period_jitter_by_session_type','Cycle-clock agreement across acquisition stages',
        'Same inputs as Figure 01, grouped by A0–B2 stage.',
        'Long-epoch HARP edge period compared to the nominal SLAP2 DMD cycle; p95 within-epoch DI3 period jitter.',
        'Identifies whether timing agreement varies systematically with experimental stage.',
        'HARP timestamp granularity contributes to measured interval jitter; irregular/off-epoch gaps excluded.')

### Real-data cycle overlays at three timescales

In each panel **pink ticks are directly measured HARP DI3 rising edges**. Green ticks are **nominal SLAP2 cycle boundaries reconstructed using the reported line clock and anchored to the first pink edge of that HARP epoch**. This is the strongest possible comparison without a second independently timestamped SLAP2 acquisition line. Small-scale overlap at the anchor is imposed; divergence over time measures nominal-rate agreement. No voltage or glutamate data are filtered or plotted here.

In [ ]:
ACTIVE_REPRESENTATIVES=[s for s in REPRESENTATIVES if s in set(selected_rows.session_id.astype(str))]
if not ACTIVE_REPRESENTATIVES:ACTIVE_REPRESENTATIVES=selected_rows.session_id.astype(str).head(3).tolist()
for sid in ACTIVE_REPRESENTATIVES:
    record,detail=focus(sid)
    if record is None:continue
    segs=detail.get('segments',[])
    chosen=max(segs,key=lambda s:s['n_pulses']) if segs else None
    fig,axs=plt.subplots(3,1,figsize=(10,7),sharey=True)
    for ax,width,position in zip(axs,[60,1,.20],[.5,.5,.5]):
        if chosen is None or not len(chosen['time_nominal_anchored']):
            missing(ax,'Requires DI3 edges and SLAP2 acquisition metadata');continue
        t=chosen['time_harp'];lo=t[0]+max(0,(t[-1]-t[0]-width)*position);hi=lo+width
        hp=t[(t>=lo)&(t<=hi)]
        nm=chosen['time_nominal_anchored'];nm=nm[(nm>=lo)&(nm<=hi)]
        ax.vlines(nm,.1,.8,color='#2b9d47',lw=1.2,label='SLAP2 nominal (DI3-anchored)')
        ax.vlines(hp,-.8,-.1,color='#d15a93',lw=1.0,label='HARP measured DI3')
        ax.set_xlim(lo,hi);ax.set_yticks([]);ax.set_xlabel(f'HARP clock seconds (window {width:g} s)')
        ax.grid(alpha=.2,axis='x')
    axs[0].legend(loc='upper right',fontsize=8,ncol=2)
    publish(fig,f'03_clock_overlay_{sid}','Clock pulse overlays · '+sid,
            record.get('path_harp_csv','')+'; '+(record.get('path_source_voltage') or record.get('path_source_glu') or ''),
            'Reconstruct SLAP2 cycle boundaries from nominal source lineRateHz and linesPerCycle; set epoch origin at first HARP DI3 edge; overlay pulse times over 60 s, 1 s, 200 ms.',
            'Demonstrates cadence coincidence and any divergence at progressively finer time scales.',
            'Nominal SLAP2 ticks are constructed, not a second recorded signal; zero lag at epoch start is defined by construction.')

### Epoch-relative residuals and continuity

For pulse ordinal `k` in each *contiguous* HARP epoch, calculate `t_DI3[k] - (t_DI3[0] + k*T_nominal)`. A flat line means that DI3 and nominal source-cycle periods agree; a slope means a nominal-rate difference, and a jump can flag an interval anomaly. This is not the physical edge-to-edge latency. Missing-pulse checks are performed **inside** pulse trains, not between different recording windows.

In [ ]:
for sid in ACTIVE_REPRESENTATIVES:
    rec,detail=focus(sid)
    if rec is None:continue
    fig,axs=plt.subplots(1,2,figsize=(11,4.1))
    segments=detail.get('segments',[])
    if not segments or not any(len(s['residual_ms']) for s in segments):
        for ax in axs:missing(ax,'Acquisition metadata or DI3 unavailable')
    else:
        for i,s in enumerate(segments):
            if len(s['residual_ms']):
                axs[0].plot((s['time_harp']-s['time_harp'][0])/60,s['residual_ms'],lw=.85,label=f'epoch {i+1}')
                gaps=np.diff(s['time_harp']); nom=s['period_nominal_s']
                if np.isfinite(nom):axs[1].hist(1000*(gaps-nom),bins=45,histtype='step',alpha=.85,label=f'epoch {i+1}')
        axs[0].legend(fontsize=8);axs[1].legend(fontsize=8)
    axs[0].set_xlabel('Minutes since first DI3 pulse of each epoch');axs[0].set_ylabel('Anchored DI3 − nominal SLAP2 (ms)')
    axs[1].set_xlabel('Observed − nominal cycle interval (ms)');axs[1].set_ylabel('Counts')
    publish(fig,f'04_drift_{sid}','Epoch-local cadence drift · '+sid,
            rec.get('path_harp_csv','')+'; source metadata in '+(rec.get('path_source_voltage') or rec.get('path_source_glu') or ''),
            'Anchor first DI3 rising pulse to nominal cycle #0 separately in each contiguous pulse train; preserve unfiltered raw time differences.',
            'Slope and long-term displacement quantify mismatch between nominal and recorded cycle rates; interval histograms show jitter.',
            'No independent SLAP2 electrical timestamp exists. Residuals cannot establish an absolute HARP-versus-SLAP2 latency.')

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(10.5,4.2))
for col,ax,xlab in [('relative_anchored_drift_end_ms',axs[0],'Largest absolute epoch-end drift (ms)'),
                     ('di3_n_irregular_intervals',axs[1],'Outlier intervals inside acquisition epochs')]:
    v=finite(df,col).dropna()
    if len(v):ax.hist(v,bins=min(30,max(5,int(np.sqrt(len(v))*3))),color='#507fa3',edgecolor='white')
    else:missing(ax,'No comparable sessions')
    ax.set_xlabel(xlab);ax.set_ylabel('Sessions')
publish(fig,'05_cadence_drift_dataset','Across-session cycle cadence drift and pulse continuity',
        'Per-session HARP DI3 rising edges and source linesPerCycle/lineRateHz; accepted acquisition epochs.',
        'Absolute end-of-epoch DI3 minus nominal source-cycle displacement after first-pulse anchoring; interval departures beyond 0.5×–1.5× nominal.',
        'Characterizes source-cadence plausibility, rate mismatch and unexpected pulse intervals without counting expected recording-window overhang as dropouts.',
        'End-point displacement is affected by nominal acquisition-rate accuracy and epoch length; not a measured electrical synchronization offset.')

### Dataset distribution of mean DI3-versus-nominal discrepancy

This answers the closest *identifiable* alternative to “mean SLAP2–HARP lag”: report the mean difference between the observed HARP DI3 rising edges and reconstructed SLAP2 nominal cycle times **after setting the first edge of each HARP epoch to zero**. The distribution characterizes accumulated clock-rate discrepancy; absolute electrical lag has no independent reference here.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(11,4.2))
for col,ax,xlabel in [('relative_anchored_discrepancy_mean_us',axs[0],'Mean anchored discrepancy (µs)'),
                     ('relative_anchored_discrepancy_p95_us',axs[1],'Mean epoch p95 absolute discrepancy (µs)')]:
    for stage,g in df.groupby('session_type'):
        v=finite(g,col).dropna()
        if len(v):ax.hist(v,bins=min(24,max(5,int(np.sqrt(len(v))*2))),alpha=.35,label=stage)
    ax.set_xlabel(xlabel);ax.set_ylabel('Sessions');ax.axvline(0,color='black',ls='--',lw=.7)
    ax.legend(fontsize=7)
publish(fig,'05b_session_relative_lag_distribution','Session distributions of anchored clock discrepancy',
        'HARP DI3 rising edges; SLAP2 source linesPerCycle and lineRateHz; registry session stage.',
        'Difference between real HARP DI3 edges and SLAP2 nominal cycle ordinal prediction; set first DI3 edge to nominal index zero in each epoch.',
        'Quantifies within-epoch rate agreement and accumulation across sessions; closest testable alternative to a mean electrical edge lag.',
        'Anchoring imposes zero initial lag. Values are NOT an independently measured SLAP2-to-HARP electrical delay.')

## 3 · Imaging epoch selection and clipping

The source acquisition duration comes from raw SLAP2 epoch `nCycles × linesPerCycle / lineRateHz`. The behavior window comes from the HARP DI3 pulse-train epochs. The processing code pairs accepted epochs chronologically and clips physiology samples when the source lasts longer than the HARP window, **without rescaling sample time**.

In the nested illustration, **the source start is placed at the first HARP DI3 pulse for illustration only** (an independent source-start timestamp does not exist). The observed duration excess is real; which fraction lies before versus after the HARP window is not directly determined by the two duration series alone. A separate panel shows the actual recorded HARP epochs on its native clock.

In [ ]:
for sid in ACTIVE_REPRESENTATIVES:
    rec,detail=focus(sid)
    if rec is None:continue
    fig,axs=plt.subplots(2,1,figsize=(11,5.7))
    epochs=detail.get('segments',[]);src=fnum(rec.get('source_duration_s'))
    hp=fnum(rec.get('harp_di3_duration_s'))
    if np.isfinite(src) and np.isfinite(hp) and src>0 and hp>0:
        axs[0].broken_barh([(0,src)],(1.1,.6),facecolors='#2b9d47',alpha=.55)
        axs[0].broken_barh([(0,hp)],(.1,.6),facecolors='#d15a93',alpha=.75)
        axs[0].axvline(hp,color='black',ls='--',lw=1)
        axs[0].text(hp,1.8,f' Δ duration = {src-hp:+.3f} s',ha='right' if hp>src*.3 else 'left',fontsize=9)
        axs[0].set_xlim(-max(src*.01,1),max(src,hp)*1.05);axs[0].set_yticks([.4,1.4],['HARP DI3','SLAP2 source'])
        axs[0].set_xlabel('Illustrative source-relative seconds (starts aligned by assumption)')
    else:missing(axs[0],'Source and HARP duration not both available')
    if epochs:
        t0=min(e['start_time'] for e in epochs)
        for i,e in enumerate(epochs):
            axs[1].broken_barh([(e['start_time']-t0,e['end_time']-e['start_time'])],(i+.1,.68),facecolors='#d15a93')
        axs[1].set_ylim(0,len(epochs)+.2);axs[1].set_yticks([i+.4 for i in range(len(epochs))],[f'DI3 epoch {i+1}' for i in range(len(epochs))])
        axs[1].set_xlabel('HARP elapsed seconds from first retained DI3 rising edge')
    else:missing(axs[1],'No HARP acquisition epoch')
    publish(fig,f'06_clipping_{sid}','Source acquisition versus HARP DI3 recording window · '+sid,
            rec.get('path_harp_csv','')+'; source metadata; analysis/qc/behavior/di3_pulse_train_detection.json where available.',
            'Measure acquisition durations independently. Upper: nested durations under an explicitly ASSUMED common start; lower: observed DI3 epoch bounds on HARP clock.',
            'Demonstrates why total source and HARP pulse counts need not match and why tail clipping is expected for longer source streams.',
            'Absolute SLAP2 start/end in HARP time not measured. Upper nesting is a visualization of durations, not independent proof of boundary registration.')

### Processing provenance: which physiology samples were actually clipped?

If postprocessing saved `source_epoch_qc`, the following panel shows **actual retained and discarded samples** by source epoch and DMD. Unlike the duration-only illustration above, the retained sample counts come directly from the pipeline's recorded reconciliation decision. These records do not independently establish the absolute start of SLAP2 on the HARP clock.

In [ ]:
for sid in ACTIVE_REPRESENTATIVES:
    rec,_=focus(sid)
    if rec is None:continue
    try:rows=json.loads(rec.get('saved_clipping_rows_json','[]'))
    except (TypeError,ValueError):rows=[]
    fig,ax=plt.subplots(figsize=(10.8,max(2.8,1.1+.50*len(rows))))
    valid=[]
    for i,r in enumerate(rows):
        length=fnum(r.get('source_duration_s'))
        kept=fnum(r.get('kept_duration_s'))
        if not np.isfinite(kept):
            original=fnum(r.get('source_samples')); saved=fnum(r.get('kept_samples'))
            if np.isfinite(original) and original>0 and np.isfinite(saved) and np.isfinite(length):
                kept=length*saved/original
        if not np.isfinite(length) or not np.isfinite(kept) or kept<0:continue
        src_path=r.get('qc_location','')
        dm='DMD1' if '/DMD1/' in src_path else 'DMD2' if '/DMD2/' in src_path else 'source'
        label=f'{dm} · epoch {r.get("source_epoch_index",i+1)}'
        valid.append((label,length,kept,r))
    if valid:
        for i,(label,length,kept,row) in enumerate(valid):
            ax.barh(i,kept,color='#2b9d47',height=.65,label='Kept' if i==0 else '')
            if length>kept:
                ax.barh(i,length-kept,left=kept,color='#d15a93',height=.65,
                        label='Discarded (tail clipping)' if i==0 else '')
            if len(valid)<=10:
                ax.text(length+max(length*.005,.2),i,f'{length-kept:.2f} s trimmed',va='center',fontsize=8)
        ax.set_yticks(range(len(valid)),[x[0] for x in valid]);ax.invert_yaxis()
        ax.set_xlabel('Source-relative duration (s) · processing kept/discarded')
        ax.set_xlim(0,max(x[1] for x in valid)*1.17);ax.legend(fontsize=8)
    else:missing(ax,'No saved source_epoch_qc rows (processing clipping not directly documented)')
    publish(fig,'06b_processing_clip_'+sid,'Actual saved per-epoch clipping decisions · '+sid,
            rec['session_dir']+'/analysis/qc/{voltage,glutamate}/*qc_metadata.json / source_epoch_qc',
            'Visualize saved source epoch duration and kept_duration_s (or proportional kept samples where unavailable), with trailing discarded source duration.',
            'Shows the exact per-DMD/epoch tail-clipping decisions recorded by the processing pipeline.',
            'Processing metadata document which samples were retained; bars are source-relative, not an independently determined SLAP2 start on HARP time.')

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(11,4.2))
valid=finite(df,'duration_overhang_s').dropna()
if len(valid):axs[0].hist(valid,bins=min(30,max(5,int(np.sqrt(len(valid))*2))),color='#507fa3',edgecolor='white')
else:missing(axs[0],'Missing source epoch counts')
axs[0].set_xlabel('Source acquisition − recorded DI3 duration (s)');axs[0].set_ylabel('Sessions')
for stage,g in df.groupby('session_type'):
    vals=finite(g,'duration_overhang_s').dropna()
    if len(vals):axs[1].scatter([stage]*len(vals),vals,alpha=.7,s=15)
axs[1].axhline(0,color='black',ls='--',lw=1)
axs[1].set_ylabel('Acquisition window duration difference (s)')
publish(fig,'07_duration_overhang_population','Acquisition-duration differences across sessions',
        'SLAP2 source metadata per epoch and HARP DI3 rising-edge segments, grouped by registry session_type.',
        'Difference between summed source nominal duration and summed measured HARP DI3 epoch duration.',
        'Reports expected additional source acquisition outside the HARP-recorded window; can identify anomalously large differences.',
        'Not the onset offset, physical clipping position, or evidence of dropped DI3 pulses. Source durations require valid epoch metadata.')

## 4 · BonVision ↔ HARP photodiode alignment (real data)

Early sessions: `frame_modclass_affine_anchored_edges` uses `Frame % modulo` candidate transitions and affine correction. Newer sessions: BV-encoded photodiode matching uses a piecewise interpolation of matched transitions. Crucially, the `alignment_method` in the event CSV takes precedence when metadata say merely `existing_corrected_timestamps`.

**Measured comparison:** corrected BV predicted transitions versus HARP analog photodiode transitions. This is a meaningful relative-timing *fit consistency check*, but the modern fitted photodiode edges are not statistically independent of the fitted warp; the legacy method additionally faces whole-cycle correspondence ambiguity. Only use the verified display refresh period to apply the final one-frame acceptance criterion.

In [ ]:
def load_behavior_events(sid,rec,detail):
    log=rec.get('path_bonsai','');pdfile=rec.get('path_pd_pkl','')
    if not (log and pdfile):return {},None
    pdfile=Path(pdfile);log=Path(log)
    if not (pdfile.exists() and log.exists()):return {},None
    saved=read_json(Path(rec['session_dir'])/'analysis'/'qc'/'behavior'/'alignment_meta.json')
    try:
        m=event_pd_diagnostics(log,pdfile,rec.get('alignment_pathway','unknown'),
             frame_period_ms=FRAME_MS,saved_phase=fnum(saved.get('phase')),
             di3_edges_abs=None,shared_origin_verified=False)
        return m,pdfile
    except Exception as e:
        print('PD diagnostic:',sid,type(e).__name__,e);return {},None

EVENTS={}; rows=[]
for sid in ACTIVE_REPRESENTATIVES:
    rec,part=focus(sid)
    if rec is None:continue
    ev,pdfile=load_behavior_events(sid,rec,part)
    EVENTS[sid]=(ev,pdfile)
    rows.append(dict(session_id=sid,alignment_pathway=rec.get('alignment_pathway'),
                     source_event_path=rec.get('path_bonsai'),
                     source_pd_path=str(pdfile) if pdfile else '',
                     n=ev.get('event_match_n',0),p95_abs_ms=ev.get('event_p95_abs_ms',np.nan),
                     fraction_within_reference_frame=ev.get('event_fraction_within_frame',np.nan),
                     event_status=ev.get('event_comparison_status','unavailable')))
display(pd.DataFrame(rows))

In [ ]:
for sid in ACTIVE_REPRESENTATIVES:
    if sid not in EVENTS:continue
    ev,pdfile=EVENTS[sid]
    fig,axs=plt.subplots(1,2,figsize=(11,4.2))
    observed=np.asarray(ev.get('event_harp_time_s',[]),float)
    residual=np.asarray(ev.get('event_residual_ms',[]),float)
    if len(observed):
        first=float(np.median(observed));width=8
        select=(observed>first-width/2)&(observed<first+width/2)
        axs[0].vlines(observed[select],0,1,color='#d15a93',label='HARP analog PD edges')
        axs[0].vlines((observed+residual/1000)[select],1.1,2.1,color='#2879b8',alpha=.7,label='BV corrected predictions')
        axs[0].set_xlim(first-width/2,first+width/2)
        axs[0].set_yticks([]);axs[0].set_xlabel('Seconds on photodiode-relative clock')
        axs[0].legend(fontsize=8)
        axs[1].hist(residual,bins=45,color='#507fa3',edgecolor='k')
        axs[1].axvline(FRAME_MS,color='black',ls='--');axs[1].axvline(-FRAME_MS,color='black',ls='--')
        axs[1].set_xlabel('BonVision predicted − HARP detected (ms)')
        axs[1].set_ylabel('Paired events')
    else:
        for a in axs:missing(a,'No corrected event/PD pairs')
    publish(fig,f'08_bv_harp_{sid}','Real BonVision/HARP event correspondence · '+sid,
            'BonVision event log corrected_timestamps + method/phase; HARP photodiode.pkl analog transitions.',
            'Monotonic nearest matched corrected predicted transitions compared to analog photodiode threshold crossings. Encoded uses rising edges; legacy uses frame-modulo edges.',
            'Overlay and residual distribution quantify fit consistency; dashed lines show assumed 60-Hz frame period.',
            'Fit-consistency is not an independent holdout; legacy periodic patterns may have cycle ambiguity; 60 Hz is a reference, not per-session verified refresh.')

In [ ]:
for sid in ACTIVE_REPRESENTATIVES:
    if sid not in EVENTS:continue
    ev,_=EVENTS[sid];t=np.asarray(ev.get('event_bv_time_s',[]),float)
    e=np.asarray(ev.get('event_residual_ms',[]),float)
    fig,ax=plt.subplots(figsize=(10.5,4.1))
    if len(e):
        ax.scatter(t/60,e,s=5,alpha=.45)
        ax.axhline(FRAME_MS,c='black',ls='--',lw=.8);ax.axhline(-FRAME_MS,c='black',ls='--',lw=.8)
        ax.axhline(0,c='.4',lw=.8)
        ax.set_xlabel('Corrected BonVision event time (min)');ax.set_ylabel('Predicted BV − measured HARP PD (ms)')
    else:missing(ax,'No corrected event residuals')
    publish(fig,f'09_bv_drift_{sid}','BonVision/HARP residual over time · '+sid,
            'Corrected BV event timestamps and HARP photodiode threshold transitions.',
            'Point-wise paired event residual versus recording time; horizontal dashed lines = nominal 60 Hz frame.',
            'Reveals time-dependent misalignment, discontinuities, and outliers that a single session median would conceal.',
            'In-sample consistency, not independent validation of absolute event correspondence.')

### Three-clock real-data raster: physical HARP photodiode, BonVision predictions, and HARP DI3

The native HARP analog photodiode and DI3 timestamps should be in a common hardware clock (subject to how the extracted pickle index was saved). This plot checks that their recorded intervals numerically overlap. It shows **real photodiode amplitudes**, HARP DI3 rising edges, and corrected BonVision predicted photodiode transitions together without resampling or smoothing. Green SLAP2 sample-clock ticks need not be included: DI3 **is** SLAP2's HARP-recorded clock.

If HARP photodiode and DI3 indices do not overlap, the notebook omits the combined plot and requests clock-origin verification; it does not silently shift either stream.

In [ ]:
for sid in ACTIVE_REPRESENTATIVES:
    rec,part=focus(sid)
    if rec is None:continue
    ev,pdfile=EVENTS.get(sid,({},None))
    fig,axs=plt.subplots(3,1,figsize=(11,6.5),sharex=True)
    edges=np.asarray(part.get('di3_edges',[]),float)
    if pdfile and len(edges) and len(ev.get('event_harp_time_s',[])):
        data=_load_pd_local(pdfile)
        tt=np.asarray(data.index,float)
        origin=float(tt[0]);end=float(tt[-1]);
        pred=origin+np.asarray(ev['event_harp_time_s'],float)+np.asarray(ev['event_residual_ms'],float)/1000
        observed=origin+np.asarray(ev['event_harp_time_s'],float)
        ok=(max(tt[0],edges[0])<min(tt[-1],edges[-1]) and
            np.isfinite(tt).all())
        if ok:
            common_start=max(tt[0],edges[0]);common_end=min(tt[-1],edges[-1]);
            inrange=observed[(observed>=common_start+2)&(observed<=common_end-2)]
            center=float(np.median(inrange)) if len(inrange) else 0.5*(common_start+common_end)
            left=center-2;right=center+2
            mask=(tt>=left)&(tt<=right)
            axs[0].plot(tt[mask]-left,np.asarray(data['AnalogInput0'])[mask],color='#d15a93',lw=.65)
            axs[0].set_ylabel('HARP PD amplitude')
            a=observed[(observed>=left)&(observed<=right)]
            bp=pred[(pred>=left)&(pred<=right)]
            axs[1].vlines(a-left,.55,1.0,color='#d15a93',label='HARP PD detected')
            axs[1].vlines(bp-left,0,.45,color='#2879b8',label='Corrected BonVision')
            axs[1].set_ylabel('Photodiode events');axs[1].legend(loc='upper right',fontsize=8)
            x=edges[(edges>=left)&(edges<=right)]
            axs[2].vlines(x-left,0,1,color='#575757',lw=.7)
            axs[2].set_ylabel('SLAP2 clock on DI3')
            for ax in axs:ax.set_yticks([]) if ax is not axs[0] else None;ax.set_xlim(0,right-left)
            axs[2].set_xlabel(f'Elapsed seconds from HARP time {left:.6f}')
        else:
            for ax in axs:missing(ax,'HARP analog PD and DI3 raw time origins do not overlap')
    else:
        for ax in axs:missing(ax,'Requires HARP PD, DI3, and corrected BV events')
    publish(fig,f'09b_three_clock_{sid}','Three-clock physical event raster · '+sid,
            str(pdfile)+'; '+rec.get('path_harp_csv','')+'; '+rec.get('path_bonsai',''),
            'AnalogInput0 raw (no smoothing), detected HARP PD transitions, corrected BV predictions and observed HARP DI3 pulses on numerically overlapping native HARP indices.',
            'Direct visual illustration of stimuli and SLAP2 acquisition cycles on HARP time, and BonVision event prediction relative to physical display changes.',
            'Source SLAP2 clock is HARP DI3 itself, not independently timestamped; correct BV mapping and photodiode extraction origin still need verification.')

### Direct HARP photodiode-to-DI3 phase measurements

Here, both event series are measured by HARP: analog photodiode edges represent physical screen transitions; DI3 rising edges are the SLAP2 acquisition clock. The signed phase to the nearest DI3 pulse is computed only if the original HARP timestamp indices overlap. **This phase is bounded by half a cycle and is not synchronization error**: a display transition may occur at any phase of the acquisition cycle. It is a useful real-data illustration that both events can be placed on the same HARP time axis.

In [ ]:
phase_rows=[]
phase_ids=df.session_id.astype(str).tolist() if ANALYZE_ALL_PHOTODIODE else ACTIVE_REPRESENTATIVES
for sid in phase_ids:
    record,detail=focus(sid)
    if record is None:continue
    pdfile=record.get('path_pd_pkl')
    if not pdfile or not Path(pdfile).is_file():continue
    try:
        phase=harp_pd_di3_phase(pdfile,detail.get('di3_edges',[]))
    except Exception as exc:
        print(sid,'PD↔DI3 error:',type(exc).__name__,exc);continue
    phase_rows.append(dict(session_id=sid,alignment_pathway=record.get('alignment_pathway'),
                           modality=record.get('indicator1'),n=phase['n_events'],
                           median_phase_ms=phase['phase_median_ms'],p95_abs_phase_ms=phase['phase_p95_abs_ms'],
                           phase_status=phase['status']))
    if sid in ACTIVE_REPRESENTATIVES:
        fig,axs=plt.subplots(1,2,figsize=(10.8,4.1))
        if phase['n_events']:
            axs[0].hist(phase['phase_ms'],bins=40,color='#507fa3',edgecolor='white')
            axs[1].scatter((phase['event_time_s']-phase['event_time_s'][0])/60,phase['phase_ms'],s=4,alpha=.5)
        else:
            for ax in axs:missing(ax,phase['status'])
        axs[0].set_xlabel('HARP screen transition − nearest DI3 rise (ms)');axs[0].set_ylabel('Physical events')
        axs[1].set_xlabel('HARP elapsed minutes');axs[1].set_ylabel('Within-cycle phase (ms)')
        publish(fig,'09c_pd_di3_phase_'+sid,'Physical stimulus-to-SLAP2-cycle phase · '+sid,
                record.get('path_harp_csv','')+'; '+str(pdfile),
                'Measured HARP photodiode threshold transitions minus nearest observed HARP DI3 rise; reject transition times near acquisition-off gaps.',
                'Reports the physical relationship between display onset and an SLAP2 acquisition cycle, on the HARP clock.',
                'Nearest-cycle phase is not absolute HARP–SLAP2 lag or sync error; photodiode.pkl and DI3 native time-origin equivalence must be confirmed.')
phase_table=pd.DataFrame(phase_rows)
if not phase_table.empty:phase_table.to_csv(OUTPUT/'harp_pd_di3_phase_by_session.csv',index=False)
fig,axs=plt.subplots(1,2,figsize=(10,4))
for label,g in phase_table.groupby('alignment_pathway') if not phase_table.empty else []:
    for ax,col in [(axs[0],'median_phase_ms'),(axs[1],'p95_abs_phase_ms')]:
        v=finite(g,col).dropna()
        if len(v):ax.hist(v,bins=min(24,max(5,int(np.sqrt(len(v))*2))),alpha=.5,label=label)
axs[0].set_xlabel('Median physical PD→DI3 cycle phase (ms)');axs[1].set_xlabel('p95 |phase| (ms)')
for ax in axs:ax.set_ylabel('Sessions');ax.legend(fontsize=8)
publish(fig,'09d_pd_di3_session_distribution','Recorded screen-transition phase within SLAP2 acquisition cycles',
        'HARP photodiode.pkl and HARP_df DI3 rising edges for sessions with available native-clock indices.',
        'Median signed event-to-nearest-clock phase and its p95 magnitude per session. Focus-only subset unless full PD scan enabled.',
        'Shows display-to-imaging hardware timing on a common recorded clock, with inter-session variability.',
        'Phase is not misalignment; absent or nonoverlapping native HARP indices are excluded rather than shifted to match.')

### Population-level BonVision/HARP distributions

The fast pass first plots saved behavior-QC `edge_rmse_intervals` (different from per-event residuals). To obtain directly measured per-event session histograms throughout the A0–B2 cohort, change `ANALYZE_ALL_PHOTODIODE=True`; those results are cached separately in the output directory. Legacy and encoded sessions remain separated. Historical missing corrected event logs are reported, not substituted with arbitrary fit quality.

In [ ]:
full_behavior=[]
if ANALYZE_ALL_PHOTODIODE:
    for _,rec in df.iterrows():
        sid=str(rec['session_id']); file=OUTPUT/'cache'/f'{sid}_behavior.json'
        if file.exists() and not FORCE_METADATA_REFRESH:
            try:full_behavior.append(json.loads(file.read_text()));continue
            except (OSError,ValueError):pass
        result,_=load_behavior_events(sid,rec,{})
        r={'session_id':sid,'alignment_pathway':rec.get('alignment_pathway'),
           'p95_abs_ms':result.get('event_p95_abs_ms',np.nan),
           'fraction_within_reference_frame':result.get('event_fraction_within_frame',np.nan),
           'coverage':result.get('event_coverage',np.nan),
           'n_matched':result.get('event_match_n',0),
           'status':result.get('event_comparison_status','not_available')}
        full_behavior.append(r);file.write_text(json.dumps(r,default=str))
else:
    full_behavior=[dict(session_id=x['session_id'],alignment_pathway=x['alignment_pathway'],
                        p95_abs_ms=x['p95_abs_ms'],fraction_within_reference_frame=x['fraction_within_reference_frame'],
                        coverage=np.nan,n_matched=x['n'],status=x['event_status']) for x in rows]
behavior=pd.DataFrame(full_behavior)
behavior.to_csv(OUTPUT/'behavior_event_diagnostics.csv',index=False)
fig,axs=plt.subplots(1,3,figsize=(12.4,4.1))
for k,g in df.groupby('alignment_pathway'):
    v=finite(g,'bv_harp_interval_rmse_ms').dropna()
    if len(v):axs[0].hist(v,bins=22,alpha=.48,label=k)
axs[0].set_xlabel('Saved interval RMSE (ms)');axs[0].set_ylabel('Sessions')
for k,g in behavior.groupby('alignment_pathway') if not behavior.empty else []:
    for ax,col in [(axs[1],'p95_abs_ms'),(axs[2],'fraction_within_reference_frame')]:
        v=finite(g,col).dropna()
        if len(v):ax.hist(v,bins=20,alpha=.55,label=k)
axs[1].axvline(FRAME_MS,ls='--',color='black',lw=1)
axs[1].set_xlabel('p95 |event residual| (ms)');axs[1].set_ylabel('Sessions')
axs[2].set_xlabel('Fraction of events within 60-Hz reference frame');axs[2].set_ylabel('Sessions')
for ax in axs:ax.legend(fontsize=7)
publish(fig,'10_population_behavior','BonVision/HARP timing distributions by alignment method',
        'Saved analysis/qc/behavior/alignment_meta.json (left); corrected BV CSV and HARP analog photodiode.pkl (middle/right).',
        'Saved HARP/BV interval RMSE shown separately from real paired-event residual p95 and within-frame fraction. Colored by original alignment method.',
        'Quantifies precision of both synchronization algorithms across eligible sessions and reveals session variability.',
        'Interval RMSE is NOT point-wise lag. Paired-event error is fitted consistency, not independent ground truth; full-event columns only cover scanned sessions.')

## 5 · Summary for Kaspar and source-provenance export

**Report separately:** DI3-to-SLAP2 nominal cycle correspondence and its drift, unmatched source duration/clipping, and BV-to-HARP physical photodiode event residuals. The *absolute* electrical SLAP2→DI3 lag remains undefined because DI3 is the SLAP2 clock measured in HARP time; do not substitute constructed coincident waveforms as a new measurement.

A frame-level synchronization argument follows the hardware chain: SLAP2 line/cycle numbering anchored to HARP DI3; HARP photodiode transitions measure physical visual display time; BonVision alignment predicts those transitions. The next high-level validation is confirming pulse-to-source epoch indexing for each modality and identifying the correct periodic legacy correspondence.

In [ ]:
assert 't03_absolute_slap2_harp_lag_ms' in df.columns
out=df[['session_id','session_type','clock_relationship','alignment_pathway',
        'di3_period_ms','source_period_ms','period_error_ppm',
        'relative_anchored_drift_end_ms','di3_n_irregular_intervals',
        'duration_overhang_s','bv_harp_interval_rmse_ms','source_read_status',
        't03_absolute_lag_status']].copy()
display(out.head(30))
out.to_csv(OUTPUT/'t03_kaspar_session_metrics.csv',index=False)
pd.DataFrame(CAPTIONS).to_csv(OUTPUT/'figure_provenance.csv',index=False)
print('Figures:',len(CAPTIONS),'in',OUTPUT/'figures')
print('Sessions with direct DMD clock cadence:', int((df.clock_relationship=='direct_dmd_cycle_clock').sum()))
print('Sessions with nominal source metadata and measured DI3:', int((finite(df,'period_error_ppm').notna()).sum()))
print('Sessions with source-vs-HARP duration comparison:',int(finite(df,'duration_overhang_s').notna().sum()))
print('No measured SLAP2–DI3 absolute lag is reported (correctly not identifiable).')